In [ ]:
!pip install faiss-cpu transformers torch

In [ ]:
import pandas as pd
import numpy as np
import faiss
import torch
import pickle
from transformers import AutoTokenizer, AutoModel



In [ ]:
# Load data
df = pd.read_csv("train.csv")

df["abstract_text"] = df["abstract_text"].fillna("").astype(str)

abstracts = (
    df.sort_values("line_number")
      .groupby("abstract_id")["abstract_text"]
      .apply(lambda x: " ".join(x))
      .reset_index()
      .rename(columns={"abstract_text": "abstract"})
)

abstracts.head()

In [ ]:
texts = abstracts["abstract"].tolist()
len(texts)


In [ ]:
texts = texts[:500]
len(texts)


In [ ]:
#load PubMedBERT
model_name = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Using device:", device)

In [ ]:
#Batch embedding function
def embed_batch(batch):
    inputs = tokenizer(
        batch,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )
    # Move inputs to GPU
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
    # Move result back to CPU
    embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()

    return embeddings



In [ ]:
#generate embeddings
embeddings = []
batch_size = 16

for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    embeddings.extend(embed_batch(batch))

embeddings = np.array(embeddings).astype("float32")

print("Embeddings shape:", embeddings.shape)


In [ ]:
# Create FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

print("Total vectors:", index.ntotal)

In [ ]:
# Save files
faiss.write_index(index, "pubmed_faiss.index")

with open("pubmed_texts.pkl", "wb") as f:
    pickle.dump(texts, f)

In [ ]:
from google.colab import files

files.download("pubmed_faiss.index")
files.download("pubmed_texts.pkl")

